# Chapter 3: Unit Group Index and Regulator Bounds

**Context:**
To apply Baker's Method, we need a maximal set of independent units and an upper bound on its index $I$. This script provides the computations for Section 3.2, the main goal of which is to prove Lemma 3.2.1, which gives a bound $I < 16\log^2|t| + 24.16$ for the set of independent units $\{\alpha, \alpha', u_{M_{t}}\}$.

**Methodology:**
Although the analytic class number formula can be combined with known results to get a bound on $I$ (dependent on $t$), the bounds obtained are too large to be computationally feasible for our purposes. This script uses:
1. **Rouché's Theorem:** Bounds $\log^2|\eta_1| + \log^2|\eta'_1| > 0.253$ by analyzing the roots of minimal quartic polynomials inside a disc.
2. **Verification for Abelian Cases:** Checks the index for the order in the abelian cases (that is, when $\mathcal{G}_{t} \simeq C_{4} \times C_{2}$, known from Proposition 3.1.1 in Section 3.1.1), where $K_t$ possesses more roots than $M_t$.

**Key Functions:**
* `verify_roots_of_unity()` & `verify_abelian_index()`: Handles the specific edge cases $t = 1/2 + 5/2\sqrt{-3}$ and $t = 5/2 + 3/2\sqrt{-11}$.
* `run_rouche_bounds()`: Computes the bounds on the regulator using Rouché's theorem.
* `verify_small_t_bound()`: Directly computes the constant $C$ used in the proof of Lemma 3.2.1 for $|t| < 10$.

In [1]:
import itertools
from itertools import product

# =============================================================================
# SHARED HELPER FUNCTIONS
# =============================================================================

def generate_imaginary_d_list(M):
    """
    Returns a list of all squarefree positive integers d such that there is 
    some non-real quadratic integer of absolute value at most M in Q(sqrt(-d)).
    """
    dlist = []
    for d in range(1, floor(4 * M**2 - 1) + 1):
        rem = d % 4
        if rem in (1, 2) and d <= M**2 and Integer(d).is_squarefree():
            dlist.append(d)
        elif rem == 3 and Integer(d).is_squarefree():
            # If d is 3 mod 4, we check up to 4*M^2 - 1
            dlist.append(d)
    return dlist


# =============================================================================
# PART 1: Roots of Unity Check (Lemma 3.2.2)
# =============================================================================

def verify_roots_of_unity():
    """
    Verifies the number of roots of unity in M_t and K_t for the specific 
    abelian cases where t = 1/2 + 5/2*sqrt(-3) and t = 5/2 + 3/2*sqrt(-11).
    """
    results = []
    # trace_norm pairs corresponding to the minimal polynomials of t
    for trace, norm in [(1, 19), (5, 31)]:
        d = (trace**2 - 4 * norm).squarefree_part()
        a = QQ(trace) / 2
        b = sqrt(QQ(trace**2 - 4 * norm) / d) / 2
        
        Lt = NumberField(x**2 + trace * x + norm, 't')
        t = Lt.gen()
        Ry = PolynomialRing(Lt, 'y')
        y = Ry.gen()
        
        gt = y**2 - t*y - 4
        ft = y**4 - t*y**3 - 6*y**2 + t*y + 1
        
        M = NumberField(gt, 'gamma')
        K = NumberField(ft, 'alpha')
        
        wM = M.number_of_roots_of_unity()
        wK = K.number_of_roots_of_unity()
        
        results.append(f"When t = {a} + {b}*sqrt({d}): M_t has {wM} roots of unity, K_t has {wK}.")
        
    return results


# =============================================================================
# PART 2: Rouché's Theorem Bounding (Lemma 3.2.6)
# =============================================================================

def find_R_max(abs1, abs2):
    """
    Finds a value of r such that |gamma1+gamma1'|*r^3 > r^4 + |gamma1*gamma1'+2*delta*delta'|*r^2 + |gamma1+gamma1'|*r + 1
    """
    var('x')
    return solve(abs1 * x**3 > x**4 + abs2 * x**2 + abs1 * x + 1, x > 0)

def find_coef_bounds(a2):
    """
    Given |gamma1*gamma1'+2*delta*delta'|, finds an appropriate value of m_gamma1.
    """
    if a2 in [2, 8, 18, 32, 50, 54, 72]:
        return None
        
    for i in range(0, 100):
        Rval = find_R_max(sqrt(i), sqrt(a2))
        if len(Rval) == 2:
            return (i, Rval[1][1].rhs().n())
            
    # Fallback to m = 44 if standard application fails
    return (44, sqrt(44)/4)

def find_root_bound2(a1bd, a2bd):
    """
    Finds a lower bound on log^2|eta1| + log^2|eta1'| for all possible values.
    """
    M_val = max(sqrt(a1bd), sqrt(a2bd))
    dlist = generate_imaginary_d_list(M_val)
    temp_regulator = 100  # Initial high bound to be overwritten
    
    for d in dlist:
        Ld = NumberField(x**2 + d, 'rootd')
        a2s = Ld.elements_of_norm(a2bd)
        
        for i in range(0, a1bd):
            a1s = Ld.elements_of_norm(i)
            rofu = Ld.roots_of_unity()
            
            for coefs in itertools.product(a1s, a2s, rofu):
                Ry = PolynomialRing(Ld, 'y')
                y = Ry.gen()
                
                # p is the minimal polynomial of a potential value of eta1*eta1'
                p = y**4 + coefs[0]*y**3 + (coefs[1] - 2*coefs[2])*y**2 + coefs[2]*coefs[0]*y + coefs[2]**2
                
                if not p.is_irreducible():
                    continue
                    
                Ktemp = Ld.extension(p, 'eta')
                if not Ktemp.is_galois_relative():
                    continue
                    
                Kauts = Ktemp.automorphisms()
                Krofu = Ktemp.roots_of_unity()
                eta = Ktemp.gen()
                
                if len(Kauts) != 4 or eta in Krofu:
                    continue
                    
                absrootvals = [abs(aut(eta)) for aut in Kauts]
                sumroots = sum(log(r)**2 for r in absrootvals) / 2
                
                # Check if Gal(Ktemp/Ld) is V4 instead of C4
                V4test = set(aut(aut(eta)) for aut in Kauts)
                if V4test == {eta}:
                    continue
                    
                if sumroots < temp_regulator:
                    temp_regulator = sumroots
                    
    return temp_regulator

def run_rouche_bounds(max_a2=104):
    """Executes the Rouché bounding loop for |gamma1*gamma1'+2*delta*delta'| <= sqrt(103)."""
    results = []
    
    for a2 in range(0, max_a2):
        print(f"\rEvaluating Rouché bounds for a2 = {a2}/{max_a2-1}...", end="", flush=True)
        
        if a2 in [2, 8, 18, 32, 50, 54, 72]:
            m_val = find_root_bound2(44, a2)
            minreg = min(m_val, float(log(sqrt(44)/4)**2))
            results.append((a2, minreg))
            continue
            
        minmax = find_coef_bounds(a2)
        if minmax is None:
            continue
            
        m_val = find_root_bound2(minmax[0], a2)
        results.append((a2, min(m_val, float(log(minmax[1])**2))))
        
    print(f"\rRouché bounds evaluation complete!{' '*30}")
    return results


# =============================================================================
# PART 3: Small |t| Verification (Lemma 3.2.1)
# =============================================================================

def verify_small_t_bound(m_bound=10):
    """
    Determines the largest value of C so that log^2|alpha| + log^2|alpha'| < log^2|t| + C 
    for all imaginary quadratic integers t with |t| < m_bound.
    """
    dlist = generate_imaginary_d_list(m_bound)
    max_C = -10
    total_d = len(dlist)
    
    for i, d in enumerate(dlist):
        print(f"\rScanning small |t| bounds for d = {d} ({i+1}/{total_d})...", end="", flush=True)
        Ld = NumberField(x**2 + d, 'rootd')
        
        for abst in range(1, m_bound**2 + 1):
            ts = Ld.elements_of_norm(abst)
            for t in ts:
                if t in ZZ:
                    continue
                    
                Ry = PolynomialRing(Ld, 'y')
                y = Ry.gen()
                f = y**4 - t*y**3 - 6*y**2 + t*y + 1
                
                if not f.is_irreducible():
                    continue
                    
                Kt = Ld.extension(f, 'alpha')
                alpha = Kt.gen()
                Kauts = Kt.automorphisms()
                
                absrootvals = [abs(aut(alpha)) for aut in Kauts]
                sumroots = sum(log(r)**2 for r in absrootvals) / 2
                
                tterm = float(sumroots - (log(sqrt(abst)))**2)
                if tterm > max_C:
                    max_C = tterm
                    
    print(f"\rSmall |t| verification complete!{' '*30}")
    return max_C


# =============================================================================
# PART 4: Index Verification in Abelian Cases (Lemma 3.2.1)
# =============================================================================

def verify_abelian_index():
    """
    Computes the index of the order <alpha, alpha', u_Mt> for the specific 
    abelian cases t = 1/2 + 5/2*sqrt(-3) and t = 5/2 + 3/2*sqrt(-11).
    """
    results = []
    
    for trace, norm in [(1, 19), (5, 31)]:
        d = (trace**2 - 4 * norm).squarefree_part()
        a = QQ(trace) / 2
        b = sqrt(QQ(trace**2 - 4 * norm) / d) / 2
        
        Lt = NumberField(x**2 + trace * x + norm, 't')
        t = Lt.gen()
        Ry = PolynomialRing(Lt, 'y')
        y = Ry.gen()
        
        gt = y**2 - t*y - 4
        ft = y**4 - t*y**3 - 6*y**2 + t*y + 1
        
        M = NumberField(gt, 'gamma')
        gamma = M.gen()
        UF = M.units()
        u1 = UF[0]
        u2 = M.automorphisms()[1](u1)
        
        e1 = Lt((u1 - u2) / (2*gamma - t))
        e2 = Lt((u1 + u2 - e1*t) / 2)
        
        K = NumberField(ft, 'alpha')
        alpha = K.gen()
        UK = K.unit_group()
        
        u = e1 * (alpha - 1/alpha) + e2
        
        alog = UK.log(alpha)
        a1log = UK.log((alpha - 1) / (alpha + 1))
        ulog = UK.log(u)
        
        A = matrix([
            [alog[1], alog[2], alog[3]], 
            [a1log[1], a1log[2], a1log[3]], 
            [ulog[1], ulog[2], ulog[3]]
        ])
        
        # Round to get the exact integer index.
        Index = round(abs(A.determinant()))
        results.append(f"For t = {a} + {b}*sqrt({d}), the order <alpha, alpha', u_M> has index {Index}.")
        
    return results


# =============================================================================
# MAIN RUNNER
# =============================================================================

if __name__ == "__main__":
    print("=== Lemma 3.2.2: Roots of Unity Verification ===")
    rou_results = verify_roots_of_unity()
    for res in rou_results:
        print(f"  • {res}")
        
    print("\n=== Lemma 3.2.1: Abelian Index Verification ===")
    index_results = verify_abelian_index()
    for res in index_results:
        print(f"  • {res}")
        
    print("\n=== Lemma 3.2.1: Small |t| Bounds Verification ===")
    max_c = verify_small_t_bound(m_bound=10)
    print(f"  • Maximum value of C found for |t| < 10: {max_c:.4f}")
    
    print("\n=== Lemma 3.2.6: Rouché's Theorem Bounding ===")

    rouche_results = run_rouche_bounds()
    print(f"  • Lowest regulator bound found: {min([r[1] for r in rouche_results]):.4f}")

=== Lemma 3.2.2: Roots of Unity Verification ===
  • When t = 1/2 + 5/2*sqrt(-3): M_t has 6 roots of unity, K_t has 30.
  • When t = 5/2 + 3/2*sqrt(-11): M_t has 2 roots of unity, K_t has 10.

=== Lemma 3.2.1: Abelian Index Verification ===
  • For t = 1/2 + 5/2*sqrt(-3), the order <alpha, alpha', u_M> has index 2.
  • For t = 5/2 + 3/2*sqrt(-11), the order <alpha, alpha', u_M> has index 1.

=== Lemma 3.2.1: Small |t| Bounds Verification ===
Small |t| verification complete!                              
  • Maximum value of C found for |t| < 10: 1.5033

=== Lemma 3.2.6: Rouché's Theorem Bounding ===
  • Note: This execution takes a significant amount of time.
Rouché bounds evaluation complete!                              
  • Lowest regulator bound found: 0.2532
